<a href="https://colab.research.google.com/github/arunsrajan/godot-docs-to-csv/blob/main/notebooks/Qwen3_5_Godot_Coding_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Godot coding assistant: Qwen3.5 with thinking, docs search and a GDScript checker

A coding assistant for GDScript and C# built from three pieces. There is no training in this notebook.

- **Thinking mode.** The model reasons step by step before answering, which helps with coding and algorithm problems. `THINKING = False` turns it off for quick API questions.
- **`search_godot_docs`.** The same docs search as the [first notebook](https://colab.research.google.com/github/arunsrajan/godot-docs-to-csv/blob/main/notebooks/Qwen3_5_2B_Godot_Docs_Tool_Calling.ipynb), over the [godot-docs-dataset](https://huggingface.co/datasets/arunsrajan/godot-docs-dataset), so the model uses real Godot 4 APIs.
- **`check_gdscript`.** Runs Godot 4.7.2's own parser (`godot --headless --check-only`) on the model's GDScript and returns the errors with line numbers, so the model can fix them before answering. It doesn't run the code.

What the checker caught when tested on mistakes small models make:

| Mistake | Caught? |
| --- | --- |
| `extends Node2D` but uses `velocity` / `move_and_slide()` | Yes |
| Syntax errors, e.g. a missing `:` | Yes |
| Godot 3 code such as `yield` | Yes ("removed in Godot 4, use await") |
| Wrong number or type of arguments, e.g. `signal.connect(handler, "x", int)` | Yes |
| `velocity.y += get_gravity() * delta` (a Vector2 added to a float) | Only with static types (`delta: float`), so the prompt asks for them |
| Logic errors (code that runs but does the wrong thing) | No: test the code in Godot |

**Runtime:** `Runtime > Change runtime type > T4 GPU` (free tier) runs the default **Qwen3.5-4B** (about 9.3 GB in 16-bit).
Thinking replies can be thousands of tokens, so a question with several tool calls can take a few minutes on a T4.
The output streams live.

## 1. Install Unsloth

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

## 2. The GDScript checker

Downloads the official Godot 4.7.2 Linux build from GitHub (about 78 MB) and checks it against the release's
SHA-512 checksums. Running the cell again skips the download.

In [ ]:
import hashlib, os, pathlib, subprocess, urllib.request, zipfile

GODOT_VERSION = "4.7.2-stable"
GODOT_DIR = pathlib.Path("godot").resolve()
GODOT = str(GODOT_DIR / f"Godot_v{GODOT_VERSION}_linux.x86_64")

if not os.path.exists(GODOT):
    release = f"https://github.com/godotengine/godot/releases/download/{GODOT_VERSION}"
    archive = f"Godot_v{GODOT_VERSION}_linux.x86_64.zip"
    zip_path, _ = urllib.request.urlretrieve(f"{release}/{archive}")
    sums = urllib.request.urlopen(f"{release}/SHA512-SUMS.txt").read().decode()
    expected = next(line.split()[0] for line in sums.splitlines() if line.endswith(archive))
    actual = hashlib.sha512(pathlib.Path(zip_path).read_bytes()).hexdigest()
    assert actual == expected, "Godot download doesn't match the official SHA-512 checksum"
    zipfile.ZipFile(zip_path).extractall(GODOT_DIR)
    os.chmod(GODOT, 0o755)
print(subprocess.run([GODOT, "--version"], capture_output = True, text = True).stdout)

`check_gdscript` writes the code into an empty Godot project and asks Godot to parse it.
Each check takes about a second. Try it on the broken script a small model wrote in the first notebook:

In [ ]:
import pathlib, re, subprocess

CHECK_PROJECT = pathlib.Path("gdscript_check").resolve()  # an empty Godot project to check scripts in
CHECK_PROJECT.mkdir(exist_ok = True)
(CHECK_PROJECT / "project.godot").write_text("config_version=5\n")
SCRIPT_ERROR = re.compile(r"SCRIPT ERROR: (?:Parse Error: )?(.*)\n\s*at: .*?:(\d+)\)")

def check_gdscript(code: str) -> str:
    """Checks a GDScript file with Godot's own parser (no code is run) and returns the errors."""
    (CHECK_PROJECT / "check.gd").write_text(code, encoding = "utf-8")
    try:
        result = subprocess.run(
            [GODOT, "--headless", "--path", str(CHECK_PROJECT), "--check-only", "--script", "res://check.gd"],
            stdout = subprocess.PIPE, stderr = subprocess.STDOUT, text = True, timeout = 60)
    except subprocess.TimeoutExpired:
        return "The check timed out."
    errors = [f"Line {line}: {message}" for message, line in SCRIPT_ERROR.findall(result.stdout)]
    if errors:
        return "Errors:\n" + "\n".join(errors)
    if result.returncode != 0:  # failed without the usual error format: show Godot's output
        return "Errors:\n" + result.stdout[-1500:]
    return "No errors found."

In [ ]:
broken = '''extends Node2D

@export var speed: float = 400.0

func _physics_process(delta: float) -> void:
    look_at(get_global_mouse_position())
    velocity = transform.x * Input.get_axis("down", "up") * speed
    move_and_slide()
'''
print(check_gdscript(broken))

## 3. The docs search tool

All columns are declared as text: `input` is always empty, so without this `datasets` can guess it is a
number and fail with `Failed to parse string: '' as a scalar of type double`.

In [ ]:
from datasets import load_dataset, Features, Value

columns = ["instruction", "input", "output", "category", "source"]
docs = load_dataset("arunsrajan/godot-docs-dataset", split = "train", keep_default_na = False,
                    features = Features({column: Value("string") for column in columns})).to_list()
print(len(docs), "rows")

In [ ]:
import math, re
from collections import Counter, defaultdict

STOPWORDS = {"the", "a", "an", "of", "in", "on", "to", "and", "or", "is", "are", "do", "does",
             "how", "what", "when", "why", "which", "i", "my", "me", "it", "for", "with", "godot",
             "explain", "about", "tell", "this", "that", "can", "you"}

def tokenize(text):
    words = [w for w in re.findall(r"[a-z0-9_]+", text.lower()) if w not in STOPWORDS]
    # "look_at" is also indexed as "look" and "at", so "look at" finds it.
    return words + [part for w in words if "_" in w for part in w.split("_") if part and part not in STOPWORDS]

class BM25:
    """Small keyword search index (BM25 ranking), no extra packages needed."""
    def __init__(self, texts, k1 = 1.5, b = 0.75):
        self.k1, self.b = k1, b
        self.postings = defaultdict(list)  # term -> [(doc_id, term_count)]
        self.lengths = []
        for doc_id, text in enumerate(texts):
            counts = Counter(tokenize(text))
            self.lengths.append(sum(counts.values()))
            for term, count in counts.items():
                self.postings[term].append((doc_id, count))
        self.avg_length = sum(self.lengths) / len(self.lengths)
        n = len(texts)
        self.idf = {term: math.log(1 + (n - len(p) + 0.5) / (len(p) + 0.5)) for term, p in self.postings.items()}

    def search(self, query, k = 3):
        scores = defaultdict(float)
        for term in set(tokenize(query)):
            for doc_id, count in self.postings.get(term, ()):
                norm = count + self.k1 * (1 - self.b + self.b * self.lengths[doc_id] / self.avg_length)
                scores[doc_id] += self.idf[term] * count * (self.k1 + 1) / norm
        return sorted(scores, key = scores.get, reverse = True)[:k]

In [ ]:
index = BM25([row["instruction"] + "\n" + row["output"] for row in docs])

In [ ]:
MAX_RESULT_CHARS = 1500
DOCS_URL = "https://docs.godotengine.org/en/latest/"

def format_results(doc_ids):
    parts = []
    for rank, doc_id in enumerate(doc_ids, 1):
        row = docs[doc_id]
        text = row["output"]
        if len(text) > MAX_RESULT_CHARS:
            text = text[:MAX_RESULT_CHARS] + " ..."
        parts.append(f"[{rank}] {DOCS_URL}{row['source']}\n{text}")
    return "\n\n".join(parts) if parts else "No results."

def search_godot_docs(query: str) -> str:
    return format_results(index.search(query, k = 3))

TOOLS = [{
    "type": "function",
    "function": {
        "name": "search_godot_docs",
        "description": "Search the official Godot Engine 4.7 documentation (class reference, tutorials and manuals). "
                       "Returns the 3 most relevant passages with their URLs.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string",
                          "description": "What to look up, e.g. 'Node2D look_at' or 'detect collisions CharacterBody2D'."},
            },
            "required": ["query"],
        },
    },
}]

Add `check_gdscript` to the tools the model can call:

In [ ]:
TOOLS.append({
    "type": "function",
    "function": {
        "name": "check_gdscript",
        "description": "Checks a complete GDScript file with the Godot 4 engine's own parser and returns the parse and "
                       "type errors with line numbers, or 'No errors found.'. It does not run the code, so it cannot "
                       "find logic errors. It cannot check C#.",
        "parameters": {
            "type": "object",
            "properties": {
                "code": {"type": "string",
                         "description": "The complete GDScript file, starting with its extends line."},
            },
            "required": ["code"],
        },
    },
})
TOOL_FUNCTIONS = {"search_godot_docs": search_godot_docs, "check_gdscript": check_gdscript}

## 4. Load the model

- `unsloth/Qwen3.5-4B` (default) fits a free T4 in 16-bit.
- `unsloth/Qwen3.5-2B` is faster but weaker at coding.
- `unsloth/Qwen3.5-9B` is stronger but needs an L4 or A100 in 16-bit. On a T4, set `load_in_4bit = True` and
  `load_in_16bit = False`, at some cost in quality.
- To use the model you fine-tuned in the first notebook, put its folder or Hub name here and set `THINKING = False`
  in section 5: it was trained without thinking.

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen3.5-4B"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = 16384,  # room for thinking, search results and several rounds of code checks
    load_in_4bit = False,
    load_in_16bit = True,
)
FastLanguageModel.for_inference(model)
# Qwen3.5 is a vision-language model; for text we only need the text tokenizer.
text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)

## 5. The assistant

The system prompt tells the model when to use each tool, and asks for statically typed GDScript so the checker
can find type errors.

In [ ]:
SYSTEM_PROMPT = (
    "You are a Godot Engine 4 coding assistant for GDScript and C#.\n"
    "- Use search_godot_docs to look up any Godot class, method, property or signal you are not sure about. "
    "Use only APIs that exist in Godot 4; never invent them.\n"
    "- Write GDScript with static types, for example `func _physics_process(delta: float) -> void:`, "
    "so the checker can find type errors.\n"
    "- Before answering with GDScript, call check_gdscript on the complete script and fix every error it reports. "
    "It only finds parse and type errors, so also check the logic yourself.\n"
    "- check_gdscript cannot check C#. For C#, follow the C# examples in the search results.\n"
    "- Give the final code in one fenced code block, then explain it briefly."
)

`ask()` runs the loop: the model thinks, then calls `search_godot_docs` or `check_gdscript` as often as it needs,
and answers when it's done. Its reasoning is kept between tool calls so it can continue its train of thought.
Qwen3.5 writes tool calls as `<tool_call><function=name><parameter=...>...</parameter></function></tool_call>`.

`verify_gdscript()` checks every GDScript block in the final answer once more, because the model may change the
code after its last check.

In [ ]:
TOOL_CALL = re.compile(r"<tool_call>\s*<function=([\w.-]+)>(.*?)</function>\s*</tool_call>", re.DOTALL)
PARAMETER = re.compile(r"<parameter=([\w.-]+)>\n?(.*?)\n?</parameter>", re.DOTALL)

def parse_tool_calls(reply):
    """Reads Qwen3.5's '<tool_call><function=name><parameter=x>...' blocks from a model reply."""
    calls = [{"name": name, "arguments": {k: v.strip() for k, v in PARAMETER.findall(body)}}
             for name, body in TOOL_CALL.findall(reply)]
    return calls, TOOL_CALL.sub("", reply).strip()


from transformers import TextStreamer

THINKING = True         # False: faster and shorter, fine for simple API questions
MAX_NEW_TOKENS = 6144   # per model reply; thinking needs room, raise it if replies stop mid-thought

def generate(messages, stream = True):
    prompt = text_tokenizer.apply_chat_template(messages, tools = TOOLS, add_generation_prompt = True,
                                                tokenize = False, enable_thinking = THINKING)
    inputs = text_tokenizer(prompt, return_tensors = "pt", add_special_tokens = False).to(model.device)
    # Qwen3.5's recommended sampling for thinking-mode coding / non-thinking text
    sampling = dict(temperature = 0.6, top_p = 0.95, top_k = 20) if THINKING else dict(temperature = 0.7, top_p = 0.8, top_k = 20)
    streamer = TextStreamer(text_tokenizer, skip_prompt = True, skip_special_tokens = True) if stream else None
    output = model.generate(**inputs, max_new_tokens = MAX_NEW_TOKENS, do_sample = True, streamer = streamer, **sampling)
    return text_tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens = True)

def split_thinking(reply):
    """Returns (reasoning, rest). With thinking on, the reply starts inside <think> and ends it with </think>."""
    if "</think>" not in reply:
        return ("", reply.strip()) if not THINKING else (reply.strip(), "")
    reasoning, rest = reply.split("</think>", 1)
    return reasoning.replace("<think>", "").strip(), rest.strip()

def run_tool(call):
    function = TOOL_FUNCTIONS.get(call["name"])
    if function is None:
        return f"Unknown tool: {call['name']}"
    try:
        return function(**call["arguments"])
    except TypeError as error:
        return f"Wrong arguments for {call['name']}: {error}"

def ask(question, max_rounds = 8, verbose = True):
    """Runs the tool loop: the model thinks, searches the docs and checks its GDScript until it answers."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question}]
    for _ in range(max_rounds):
        reasoning, rest = split_thinking(generate(messages, stream = verbose))
        calls, content = parse_tool_calls(rest)
        if not calls:
            return content or "(No answer: the model ran out of tokens while thinking. Raise MAX_NEW_TOKENS.)"
        # Keeping the reasoning lets the model continue its train of thought after each tool result.
        messages.append({"role": "assistant", "content": content, "reasoning_content": reasoning,
                         "tool_calls": [{"type": "function", "function": call} for call in calls]})
        for call in calls:
            result = run_tool(call)
            if verbose:
                print(f"\n----- [tool] {call['name']} -> {result[:500]}\n")
            messages.append({"role": "tool", "content": result})
    return "(Stopped: too many tool calls.)"

GDSCRIPT_BLOCK = re.compile(r"```(?:gdscript|gd)\n(.*?)```", re.DOTALL)

def verify_gdscript(answer):
    """Runs check_gdscript on every GDScript block in a final answer (the model may skip its last check)."""
    for number, code in enumerate(GDSCRIPT_BLOCK.findall(answer), 1):
        print(f"GDScript block {number}: {check_gdscript(code)}")

## 6. Try it

Each task runs the full loop, streaming the model's thinking and tool calls, then checks the final GDScript.
Change the index to try the other tasks. The C# task isn't checked: `check_gdscript` only handles GDScript.

In [ ]:
CODING_TASKS = [
    "Write a GDScript for a CharacterBody2D platformer character: move left and right with the arrow keys, "
    "jump with the ui_accept action, and allow one double jump in the air.",
    "Write a GDScript function that returns the shortest path between two cells of a 10x10 grid with walls, "
    "using AStarGrid2D. The walls are given as an Array of Vector2i.",
    "Write a GDScript Resource class for an inventory that stores item names with counts, "
    "with add_item, remove_item and has_item methods.",
    "Write a C# script for a CharacterBody2D platformer character that can double jump.",
]

answer = ask(CODING_TASKS[0])
print("=" * 80, "\n" + answer)
verify_gdscript(answer)

Ask your own question:

In [ ]:
answer = ask("How do I spawn an enemy scene every 2 seconds at a random position on screen?")
print("=" * 80, "\n" + answer)
verify_gdscript(answer)

## Limits

- **The checker finds parse and type errors, not logic errors.** Code that passes can still do the wrong thing;
  run it in Godot to test its behavior.
- **Untyped code hides type errors.** `velocity.y += get_gravity() * delta` only fails the check when `delta` is
  typed, which is why the prompt asks for static types.
- **Each script is checked on its own.** Code that uses `class_name` classes from other files of your project, or
  your autoloads, will be reported as using undeclared names.
- **C# isn't checked.** That would need Godot's .NET build and `dotnet build` on a C# project.
- **Thinking is slow on a T4.** Set `THINKING = False` for quick API questions, or raise `MAX_NEW_TOKENS` if a reply
  stops mid-thought.